In [4]:
# ============================================================
# Cell 1: Environment setup
# ============================================================

from pathlib import Path

import ee
import geemap

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

ADMIN_DIR = RAW_DIR / "admin"
H3_DIR = PROCESSED_DIR / "h3"

ADMIN_DIR.mkdir(parents=True, exist_ok=True)
H3_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Earth Engine
# ------------------------------------------------------------------

EE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# ------------------------------------------------------------------
# Pilot configuration
# ------------------------------------------------------------------

COUNTRY = "Kenya"
ADMIN1 = "Nairobi"

H3_RESOLUTION = 10

OUTPUT_PARQUET = H3_DIR / f"pilot_nairobi_h3_res{H3_RESOLUTION}.parquet"

print("=" * 60)
print("ZORYN PILOT")
print("=" * 60)
print(f"Project Root : {PROJECT_ROOT}")
print(f"Country      : {COUNTRY}")
print(f"Admin Level1 : {ADMIN1}")
print(f"H3 Resolution: {H3_RESOLUTION}")
print(f"Output       : {OUTPUT_PARQUET}")
print("=" * 60)

ZORYN PILOT
Project Root : c:\Projects\ews\zoryn
Country      : Kenya
Admin Level1 : Nairobi
H3 Resolution: 10
Output       : c:\Projects\ews\zoryn\data\processed\h3\pilot_nairobi_h3_res10.parquet


In [5]:
# ============================================================
# Cell 2: Load Nairobi County boundary
# ============================================================

import geopandas as gpd

# ------------------------------------------------------------------
# GAUL Level 1 Administrative Boundaries
# ------------------------------------------------------------------

admin1 = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filter(ee.Filter.eq("ADM0_NAME", COUNTRY))
    .filter(ee.Filter.eq("ADM1_NAME", ADMIN1))
)

count = admin1.size().getInfo()

if count != 1:
    raise ValueError(
        f"Expected exactly one feature for {ADMIN1}, found {count}."
    )

feature = admin1.first()

# ------------------------------------------------------------------
# Convert to GeoDataFrame
# ------------------------------------------------------------------

gdf = geemap.ee_to_gdf(admin1)

gdf = gdf.to_crs(4326)

# ------------------------------------------------------------------
# Save locally
# ------------------------------------------------------------------

boundary_path = ADMIN_DIR / "nairobi_county.geoparquet"

gdf.to_parquet(boundary_path)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("=" * 60)
print("Boundary Loaded")
print("=" * 60)
print(f"Features      : {len(gdf)}")
print(f"CRS           : {gdf.crs}")
print(f"Bounds        : {gdf.total_bounds}")
print(f"Saved         : {boundary_path}")
print("=" * 60)

display(gdf.head())

Boundary Loaded
Features      : 1
CRS           : EPSG:4326
Bounds        : [36.66605984 -1.4428536  37.10557557 -1.15741764]
Saved         : c:\Projects\ews\zoryn\data\raw\admin\nairobi_county.geoparquet


,geometry,ADM0_CODE,ADM0_NAME,ADM1_CODE,ADM1_NAME,DISP_AREA,EXP1_YEAR,STATUS,STR1_YEAR,Shape_Area,Shape_Leng
0,"POLYGON ((36.66606 -1.31742, 36.67514 -1.32215...",133,Kenya,51328,Nairobi,NO,3000,Member State,1000,0.05658,1.664051


In [6]:
# ============================================================
# Cell 3: Generate H3 grid
# ============================================================

import h3
import pandas as pd
import geopandas as gpd

from shapely.geometry import Polygon

# ------------------------------------------------------------------
# Load boundary
# ------------------------------------------------------------------

boundary = gpd.read_parquet(boundary_path)

geometry = boundary.geometry.iloc[0]

# ------------------------------------------------------------------
# Convert to GeoJSON for H3
# ------------------------------------------------------------------

geojson = geometry.__geo_interface__

# ------------------------------------------------------------------
# Generate H3 cells
# ------------------------------------------------------------------

h3_cells = h3.geo_to_cells(
    geojson,
    res=H3_RESOLUTION,
)

print(f"Generated {len(h3_cells):,} H3 cells.")

# ------------------------------------------------------------------
# Build GeoDataFrame
# ------------------------------------------------------------------

records = []

for cell in sorted(h3_cells):

    lat, lon = h3.cell_to_latlng(cell)

    boundary_coords = h3.cell_to_boundary(cell)

    polygon = Polygon(
        [(lng, lat) for lat, lng in boundary_coords]
    )

    records.append(
        {
            "h3_id": cell,
            "latitude": lat,
            "longitude": lon,
            "geometry": polygon,
        }
    )

h3_gdf = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326",
)

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------

h3_gdf.to_parquet(OUTPUT_PARQUET)

print("=" * 60)
print("H3 GRID")
print("=" * 60)
print(f"Resolution : {H3_RESOLUTION}")
print(f"Cells      : {len(h3_gdf):,}")
print(f"Saved      : {OUTPUT_PARQUET}")
print("=" * 60)

display(h3_gdf.head())

Generated 42,319 H3 cells.
H3 GRID
Resolution : 10
Cells      : 42,319
Saved      : c:\Projects\ews\zoryn\data\processed\h3\pilot_nairobi_h3_res10.parquet


,h3_id,latitude,longitude,geometry
0,8a7a6325b20ffff,-1.294257,37.073878,"POLYGON ((37.07434 -1.29482, 37.07459 -1.29416..."
1,8a7a6325b22ffff,-1.295479,37.074090,"POLYGON ((37.07455 -1.29604, 37.0748 -1.29538,..."
2,8a7a6325b247fff,-1.294559,37.071744,"POLYGON ((37.07221 -1.29512, 37.07246 -1.29446..."
3,8a7a6325b24ffff,-1.294099,37.070571,"POLYGON ((37.07103 -1.29466, 37.07128 -1.294, ..."
4,8a7a6325b257fff,-1.293797,37.072705,"POLYGON ((37.07317 -1.29436, 37.07342 -1.2937,..."
